# Demo: ATLAS Media Chart Builder

This notebook shows a simple end-to-end example for workshop users.

## Step 1: Imports and setup

If you use your own file, replace the file path in Step 2.

In [ ]:
from pathlib import Path
import pandas as pd

import config
from src.charts import export_charts
from src.exporters import export_cleaned_excel, export_powerpoint, export_summary_text
from src.insights import generate_summary_text
from src.quality import build_quality_notes
from src.standardizer import standardize_data


## Step 2: Load sample data (swap this with your own file)

Place your workbook in the `input/` folder and update `config.py` if needed.

In [ ]:
sample_path = config.INPUT_FOLDER / config.DEFAULT_INPUT_FILENAME
sample_path.parent.mkdir(parents=True, exist_ok=True)

if not sample_path.exists():
    demo_df = pd.DataFrame({
        'Year': [2022, 2022, 2023, 2023, 2024, 2024],
        'Media Type': ['TV', 'Digital', 'TV', 'Digital', 'TV', 'Digital'],
        'Spend': [100, 120, 110, 150, 95, 170]
    })
    with pd.ExcelWriter(sample_path, engine='openpyxl') as writer:
        demo_df.to_excel(writer, sheet_name=config.DEFAULT_SHEET_NAME, index=False)

raw_df = pd.read_excel(sample_path, sheet_name=config.DEFAULT_SHEET_NAME)
raw_df.head()


## Step 3: Standardize data

In [ ]:
standardized_df = standardize_data(
    raw_df,
    config.DEFAULT_YEAR_COLUMN,
    config.DEFAULT_CATEGORY_COLUMN,
    config.DEFAULT_VALUE_COLUMN
)
standardized_df.head()


## Step 4: Data quality notes

In [ ]:
notes_df = build_quality_notes(raw_df, standardized_df)
notes_df


## Step 5: Export charts

In [ ]:
chart_paths = export_charts(standardized_df, config.CHARTS_FOLDER, config.BRAND_COLORS, config.CHART_FIGSIZE)
chart_paths


## Step 6: Generate and edit summary text

In [ ]:
summary_text = generate_summary_text(standardized_df)
print(summary_text)


## Step 7: Export workbook, text, and PowerPoint

In [ ]:
excel_path = export_cleaned_excel(
    standardized_df, notes_df, config.REPORTS_FOLDER / config.CLEANED_WORKBOOK_FILENAME
)
txt_path = export_summary_text(summary_text, config.TEXT_FOLDER / config.SUMMARY_TEXT_FILENAME)
ppt_path = export_powerpoint(chart_paths, summary_text, config.POWERPOINT_FOLDER / config.POWERPOINT_FILENAME)

print('Excel:', excel_path)
print('Text:', txt_path)
print('PPT:', ppt_path)
